# V033r1 evaluation review

Native inspiratory mean MAE controls selection. Median and diagnostics are secondary. Run only after independent completed-artifact audit.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
root=Path.cwd();o=root/"outputs/controller_bank_screen_v033r1";s=json.loads((o/"summary.json").read_text());audit=json.loads((root/"research/controller_bank_review_v033r1.json").read_text());assert audit["status"]=="passed"
print(pd.Series({k:s[k] for k in ["MAE","reference_A_MAE","gain","bootstrap_95","eligible","bank_size","distance_confident_breaths","gate_points","changed_points","improved_changed_points","worsened_changed_points","runtime_seconds"]}).to_string())
r=pd.read_csv(o/"RC_errors.csv");r["setting"]="R="+r.R.astype(str)+", C="+r.C.astype(str);r=r.sort_values("total_absolute_error",ascending=False)
fig,axes=plt.subplots(1,2,figsize=(13,5));axes[0].barh(r.setting,r.total_absolute_error,color="#246a95");axes[0].invert_yaxis();axes[0].set(xlabel="Total absolute pressure error",title="Groups contributing most to the score")
axes[1].barh(r.setting,r.delta_total_absolute_error,color=np.where(r.delta_total_absolute_error<0,"#34816a","#b4584e"));axes[1].invert_yaxis();axes[1].axvline(0,color="black",linewidth=.8);axes[1].set(xlabel="Candidate minus baseline total error",title="Effect of the fixed controller correction")
fig.suptitle("V033r1 original grouped A validation — pressure labels used only for evaluation");fig.tight_layout();fig.savefig(o/"RC_error_contributions.png",dpi=160);plt.close(fig)
df=pd.read_csv(root/"data/train.csv",usecols=["id","breath_id","u_out","pressure"],dtype={"pressure":"float32"});bids=df.breath_id.to_numpy().reshape(-1,80)[:,0];keep=np.flatnonzero(np.isin(bids,np.load(o/"A_breath_ids.npy")));phase=df.u_out.to_numpy().reshape(-1,80)[keep];y=df.pressure.to_numpy().reshape(-1,80)[keep].astype(np.float64);mask=phase==0
base=np.zeros_like(y);base[mask]=np.load(root/"outputs/postprocessing_screen_v016/weighted_median_midpoint_then_nearest_training_pressure_codebook_A_predictions.npy");pred=np.zeros_like(y);pred[mask]=np.load(o/"A_predictions.npy");err=np.where(mask,abs(pred-y),0.);changed=np.any(pred!=base,axis=1);order=np.argsort(-err.sum(1),kind="stable");chosen=[i for i in order if changed[i]][:4]
if len(chosen)<4:chosen+= [i for i in order if i not in chosen][:4-len(chosen)]
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,i in zip(axes.ravel(),chosen):
 t=np.flatnonzero(mask[i]);ax.plot(t,y[i,t],label="Actual",color="black",linewidth=1.6);ax.plot(t,base[i,t],label="Accepted baseline",color="#246a95",alpha=.8);ax.plot(t,pred[i,t],label="Controller candidate",color="#c26131",linestyle="--");ax.set(title=f"Breath {bids[keep[i]]}: total error {err[i].sum():.2f}",xlabel="Position in breath",ylabel="Pressure")
axes[0,0].legend(fontsize=8);fig.suptitle("Difficult validation breaths: changed breaths prioritized for diagnosis");fig.tight_layout();fig.savefig(o/"difficult_breaths.png",dpi=160);plt.close(fig)
print("Saved RC_error_contributions.png and difficult_breaths.png. These plots do not change the frozen recipe or promotion rule.")


MAE                                                                  0.182879
reference_A_MAE                                                      0.182922
gain                                                                 0.000043
bootstrap_95                  [-0.0001669410228488953, 7.674560249862751e-05]
eligible                                                                False
bank_size                                                               21136
distance_confident_breaths                                               3365
gate_points                                                             15770
changed_points                                                          10742
improved_changed_points                                                  5434
worsened_changed_points                                                  5308
runtime_seconds                                                      37.71033
Saved RC_error_contributions.png and difficult_breaths.png. Thes